In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
import joblib

pd.set_option("display.max_columns", None)

In [ ]:
df = pd.read_csv("../data/traffic_events.csv")

print(df.shape)
df.head()

In [ ]:
df["is_obstruction_event"] = (
    df["event_cause"]
    .eq("vehicle_breakdown")
    .astype(int)
)

df["is_obstruction_event"].value_counts()

In [ ]:
df["start_datetime"] = pd.to_datetime(df["start_datetime"], errors="coerce")
df["end_datetime"] = pd.to_datetime(df["end_datetime"], errors="coerce")

df["duration_minutes"] = (
    df["end_datetime"] - df["start_datetime"]
).dt.total_seconds()/60

df["hour"] = df.start_datetime.dt.hour
df["day_of_week"] = df.start_datetime.dt.dayofweek

df.head()

In [ ]:
risk_df = (
    df[df.is_obstruction_event == 1]
    .groupby(["junction","zone","hour"])
    .agg(
        event_frequency=("id","count"),
        avg_duration=("duration_minutes","mean")
    )
    .reset_index()
    .fillna(0)
)

risk_df["peak_hour"] = risk_df["hour"].apply(
    lambda x: 1 if (8 <= x <= 11) or (17 <= x <= 21) else 0
)

risk_df.head()

In [ ]:
scaler = MinMaxScaler()

cols = ["event_frequency","avg_duration","peak_hour"]

risk_df[[c+"_scaled" for c in cols]] = scaler.fit_transform(
    risk_df[cols]
)

risk_df["risk_score"] = (
    0.60*risk_df.event_frequency_scaled +
    0.25*risk_df.avg_duration_scaled +
    0.15*risk_df.peak_hour_scaled
)*100

risk_df.sort_values("risk_score", ascending=False).head(10)

In [ ]:
def recommend(row):
    if row.risk_score >= 70:
        return f"Deploy response team at {int(row.hour)-1}:30"
    elif row.risk_score >= 40:
        return "Increase monitoring"
    return "Normal patrol"

risk_df["recommendation"] = risk_df.apply(recommend, axis=1)

risk_df[["junction","risk_score","hour","recommendation"]].sort_values(
    "risk_score", ascending=False
).head(10)

In [ ]:
ml_df = (
    df[df.is_obstruction_event == 1]
    .groupby(["junction","zone","hour","day_of_week"])
    .agg(
        event_frequency=("id","count"),
        avg_duration=("duration_minutes","mean")
    )
    .reset_index()
    .fillna(0)
)

ml_df[[c+"_scaled" for c in ["event_frequency","avg_duration"]]] = (
    MinMaxScaler().fit_transform(
        ml_df[["event_frequency","avg_duration"]]
    )
)

ml_df["risk_score"] = (
    0.7*ml_df.event_frequency_scaled +
    0.3*ml_df.avg_duration_scaled
)*100

ml_df.head()

In [ ]:
X = pd.get_dummies(
    ml_df.drop("risk_score", axis=1)
)

y = ml_df["risk_score"]

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

model = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05
)

model.fit(X_train,y_train)

pred=model.predict(X_test)

print("MAE:", mean_absolute_error(y_test,pred))

In [ ]:
joblib.dump(model,"../models/risk_model.pkl")
joblib.dump(X.columns,"../models/model_columns.pkl")

print("Model saved")